# 03 · Leaderboard & mCE (Results §4 / RQ1)
Clean vs robustness, and the pre-registered ρ test with a record-level bootstrap CI.

In [ ]:
import os
# make relative paths work whether launched from repo root or notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

In [ ]:
import pandas as pd
from src.analysis import (clean_leaderboard, mean_corruption_error, relative_mce,
                          rho_clean_vs_robustness, bootstrap_rho, MODEL_ORDER)
grid = pd.read_csv('results/grid.csv')
models = [m for m in MODEL_ORDER if m in set(grid.model)]
clean = clean_leaderboard(grid)
mce = mean_corruption_error(grid, 'minirocket')
rce = relative_mce(grid, 'minirocket')
lb = pd.DataFrame({'clean_auroc': pd.Series(clean), 'mCE': mce, 'relative_mCE': rce}).loc[models]
lb.sort_values('mCE').round(4)

In [ ]:
rho, p = rho_clean_vs_robustness(clean, mce.to_dict(), models)
corrs = list(grid[grid.corruption != 'clean'].corruption.unique())
lo, hi, _ = bootstrap_rho('results/preds', models, corrs, 'minirocket', n_boot=300)
print(f'Spearman rho (clean vs -mCE) = {rho:.3f} (p={p:.3f}); 95% CI [{lo:.3f}, {hi:.3f}]')
print('Pre-registered rho<0.7 & CI excludes 0.9  =>',
      'SUPPORTED' if (rho < 0.7 and hi < 0.9) else 'NOT supported')